In [ ]:
import litellm
import pandas as pd 
from src.prompts import BASE_PROMPT
from sklearn.metrics import classification_report
import numpy as np
from src.metrics import recall_at_k, page_score_np
from src.utils import load_pdf_with_plumber
from src.retrievers import get_embeddings, create_nmslib_index, ann_search, bm25_search, create_bm25_retriever
import os
from tqdm import tqdm
import dotenv
dotenv.load_dotenv(".env")

df = pd.read_csv("data/dev_questions.csv")

df['options'] = df[['A', 'B', 'C', 'D', 'E', 'F']].apply(
    lambda row: ' '.join([f"{chr(65 + i)}. {option}" for i, option in enumerate(row)]), axis=1
)

MODEL = "Qwen/Qwen3-VL-8B-Instruct-FP8"
BASE_URL = "http://localhost:8000/v1"
API_KEY = "your_api_key_here"
llm = litellm.OpenAI( base_url=BASE_URL, api_key=API_KEY )


/home/nazara/UCU/NLP/ASSIGNMENT_LAST/src/prompts.py:36: SyntaxWarning: invalid escape sequence '\{'
  COT_PROMPT = """
Your CPU supports instructions that this binary was not compiled to use: SSE3 SSE4.1 SSE4.2 AVX AVX2
For maximum performance, you can install NMSLIB from sources 
pip install --no-binary :all: nmslib
/home/nazara/UCU/NLP/ASSIGNMENT_LAST/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Baseline zero-shot 

In [7]:
answers = []
for index, row in df.iterrows():
    question = row["Question"]
    options = row["options"]
    prompt = BASE_PROMPT.format(question=question, options=options)
    response = llm.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}]
        )
    answers.append(response.choices[0].message.content)
print(classification_report(answers, df["Correct_Answer"]))


              precision    recall  f1-score   support

           A       0.65      0.59      0.62        86
           B       0.73      0.49      0.59        99
           C       0.69      0.60      0.64        81
           D       0.60      0.72      0.65        64
           E       0.54      0.72      0.62        58
           F       0.53      0.64      0.58        73

    accuracy                           0.62       461
   macro avg       0.62      0.63      0.62       461
weighted avg       0.63      0.62      0.62       461



# 2. Embedding model comparison

In [2]:
EMBEDDING_MODEL1= "paraphrase-multilingual-MiniLM-L12-v2"
EMBEDDING_MODEL2= "intfloat/multilingual-e5-base"

dirs = ["data/domain_1/dev", "data/domain_2/dev"]

pages = []
doc_id = 0
filename_to_doc_id = {}  # Mapping from filename to doc_id

for dir in dirs:
    for filename in os.listdir(dir):
        if filename.endswith(".pdf"):
            result = load_pdf_with_plumber(pdf_path=os.path.join(dir, filename), doc_id=doc_id)
            pages.extend(result)
            filename_to_doc_id[filename] = doc_id
            doc_id += 1

print(f"Loaded {len(pages)} pages from {doc_id} documents.")

df_corpus = pd.DataFrame(pages, columns=['Doc_ID', 'Page_Num', 'Text_Content'])

Loaded 1121 pages from 41 documents.


In [ ]:
from sentence_transformers import SentenceTransformer

emb_model1 = SentenceTransformer(EMBEDDING_MODEL1, trust_remote_code=True)
emb_model2 = SentenceTransformer(EMBEDDING_MODEL2, trust_remote_code=True)

pages_embeddings = get_embeddings(df_corpus['Text_Content'].tolist(), model=emb_model1)
pages_embeddings2 = get_embeddings(df_corpus['Text_Content'].tolist(), model=emb_model2)


Batches: 100%|██████████| 71/71 [00:09<00:00,  7.26it/s]


In [ ]:
nmslib_index1 = create_nmslib_index(pages_embeddings)
nmslib_index2 = create_nmslib_index(pages_embeddings2)

In [5]:
k_neighbors = 10
# ANN search with first embedding model
ann_search_results1 = {}
for query in tqdm(df['Question'].tolist(), desc="ANN Search Model 1"):
    indices, distances = ann_search(query, nmslib_index1, emb_model1, top_k=k_neighbors)
    ann_search_results1[query] = {'indices': indices, 'distances': distances}

# ANN search with second embedding model
ann_search_results2 = {}
for query in tqdm(df['Question'].tolist(), desc="ANN Search Model 2"):
    indices, distances = ann_search(query, nmslib_index2, emb_model2, top_k=k_neighbors)
    ann_search_results2[query] = {'indices': indices, 'distances': distances}

ANN Search Model 2: 100%|██████████| 461/461 [00:03<00:00, 144.25it/s]


In [ ]:
retriever = create_bm25_retriever(df_corpus['Text_Content'].tolist())

bm25_search_results = {}
k_neighbors = 10
for query in tqdm(df['Question'].tolist()):
    indices = bm25_search(query, retriever, top_k=k_neighbors)
    bm25_search_results[query] = {'indices': indices}

number_pages_doc = df_corpus.Doc_ID.value_counts().to_dict()

100%|██████████| 461/461 [00:01<00:00, 324.99it/s]     


In [7]:
def compute_metrics(ann_search_results, bm25_search_results, df_questions, df_chunked, n_pages_dict):
    ann_metrics_doc = {}
    bm25_metrics_doc = {}
    ann_pages_metric = []
    bm25_pages_metric = []

    for _, row in df_questions.iterrows():
        question = row["Question"]
        answers = "\n".join([f"{letter} - {row[letter]}" for letter in ["A", "B", "C", "D", "E", "F"]])
        correct_answer = row["Correct_Answer"]
        # Convert filename to doc_id
        correct_doc = filename_to_doc_id[row["Doc_ID"]]
        page_num = int(row["Page_Num"])

        # Documents retrieval evaluation:
        ann_retrieved_docs = [df_chunked.iloc[i]["Doc_ID"] for i in ann_search_results[question]["indices"]]
        bm25_retrieved_docs = [df_chunked.iloc[i]["Doc_ID"] for i in bm25_search_results[question]["indices"]]

        for i in [1, 3, 5, 10]:
            ann_metrics_doc[i] = ann_metrics_doc.get(i, [])
            ann_metrics_doc[i].append(recall_at_k(ann_retrieved_docs, correct_doc, i))
            bm25_metrics_doc[i] = bm25_metrics_doc.get(i, [])
            bm25_metrics_doc[i].append(recall_at_k(bm25_retrieved_docs, correct_doc, i))

        # Pages retrieval evaluation (evaluation only for first retrieved_doc):
        ann_retrieved_pages = [df_chunked.iloc[i]["Page_Num"] for i in ann_search_results[question]["indices"]]
        bm25_retrieved_pages = [df_chunked.iloc[i]["Page_Num"] for i in bm25_search_results[question]["indices"]]

        ann_page = df_chunked.iloc[ann_search_results[question]["indices"][0]]["Page_Num"]
        bm25_page = df_chunked.iloc[bm25_search_results[question]["indices"][0]]["Page_Num"]
        first_doc_correct_ann = ann_retrieved_docs[0] == correct_doc
        first_doc_correct_bm25 = bm25_retrieved_docs[0] == correct_doc

        ann_pages_metric.append(
            page_score_np(
                ann_page,
                page_num,
                n_pages_dict[correct_doc],
                first_doc_correct_ann
            )
        )
        bm25_pages_metric.append(
            page_score_np(
                bm25_page,
                page_num,
                n_pages_dict[correct_doc],
                first_doc_correct_bm25
            )
        )

    print("Performance for documents retrieval: ")
    for i in [1, 3, 5, 10]:
        print(f"ANN Recall@{i}:", np.mean(ann_metrics_doc[i]), f", BM25 Recall@{i}:", np.mean(bm25_metrics_doc[i]))
    
    print("\nPerformance for pages retrieval: ")
    print(f"ANN Page Score: {np.mean(ann_pages_metric)}, BM25 Page Score: {np.mean(bm25_pages_metric)}")
    return {
        'ann_recall@1': np.mean(ann_metrics_doc[1]),
        'ann_recall@3': np.mean(ann_metrics_doc[3]),
        'ann_recall@5': np.mean(ann_metrics_doc[5]),
        'ann_recall@10': np.mean(ann_metrics_doc[10]),
        'bm25_recall@1': np.mean(bm25_metrics_doc[1]),
        'bm25_recall@3': np.mean(bm25_metrics_doc[3]),
        'bm25_recall@5': np.mean(bm25_metrics_doc[5]),
        'bm25_recall@10': np.mean(bm25_metrics_doc[10]),
        'ann_page_score': np.mean(ann_pages_metric),
        'bm25_page_score': np.mean(bm25_pages_metric),
    }

In [8]:
# Compute metrics for both embedding models
print(f"=== Metrics for {EMBEDDING_MODEL1} ===")
compute_metrics(ann_search_results1, bm25_search_results, df, df_corpus, number_pages_doc)

print("\n" + "="*60 + "\n")

print(f"=== Metrics for {EMBEDDING_MODEL2} ===")
metrics_baseline = compute_metrics(ann_search_results2, bm25_search_results, df, df_corpus, number_pages_doc)

=== Metrics for paraphrase-multilingual-MiniLM-L12-v2 ===
Performance for documents retrieval: 
ANN Recall@1: 0.4685466377440347 , BM25 Recall@1: 0.7657266811279827
ANN Recall@3: 0.6507592190889371 , BM25 Recall@3: 0.8568329718004338
ANN Recall@5: 0.720173535791757 , BM25 Recall@5: 0.8893709327548807
ANN Recall@10: 0.8134490238611713 , BM25 Recall@10: 0.9349240780911063

Performance for pages retrieval: 
ANN Page Score: 0.37595064765757996, BM25 Page Score: 0.6681763516810793


=== Metrics for intfloat/multilingual-e5-base ===
Performance for documents retrieval: 
ANN Recall@1: 0.8763557483731019 , BM25 Recall@1: 0.7657266811279827
ANN Recall@3: 0.9414316702819957 , BM25 Recall@3: 0.8568329718004338
ANN Recall@5: 0.96529284164859 , BM25 Recall@5: 0.8893709327548807
ANN Recall@10: 0.9783080260303688 , BM25 Recall@10: 0.9349240780911063

Performance for pages retrieval: 
ANN Page Score: 0.741937855669413, BM25 Page Score: 0.6681763516810793


# 3. Chunking strategy optimization

In [9]:
from src.chunking import (
    sentence_chunker,
    fixed_size_chunking,
    recursive_character_text_splitter,
    semantic_sentence_chunker,
    sliding_window_chunker,
    hybrid_chunker,
    late_chunking
)
from functools import partial


def create_chunked_df(df_corpus, chunking_strategy) -> pd.DataFrame:
    """Create chunked dataframe from corpus using given strategy."""
    rows = [
        {'Doc_ID': row.Doc_ID, 'Page_Num': row.Page_Num, 'Text_Content': chunk}
        for row in df_corpus.itertuples()
        for chunk in chunking_strategy(row.Text_Content)
        if chunk.strip()  # Skip empty chunks
    ]
    return pd.DataFrame(rows)


# Define all chunking strategies to test
chunking_strategies = {
    # Original strategies
    "sentence_300": partial(sentence_chunker, max_length=300, n_sentences=6),
    "fixed_300": partial(fixed_size_chunking, chunk_size=300, overlap=50),
    "fixed_600": partial(fixed_size_chunking, chunk_size=600, overlap=100),
    "recursive_500": partial(recursive_character_text_splitter, max_length=500),
    
    # Improved strategies
    "semantic_500": partial(semantic_sentence_chunker, max_length=500, min_length=100),
    "sliding_500": partial(sliding_window_chunker, chunk_size=500, overlap_ratio=0.2),
    "hybrid_400_600": partial(hybrid_chunker, target_size=400, max_size=600, min_size=100),
    "late_500": partial(late_chunking, chunk_size=500, context_window=100),
}

# Create all chunked dataframes
print("Creating chunked dataframes...")

df_sentence_chunks = create_chunked_df(
    df_corpus, partial(sentence_chunker, max_length=300, n_sentences=6)
)
print(f"Sentence chunks: {len(df_sentence_chunks)}")

df_fixed_300 = create_chunked_df(
    df_corpus, partial(fixed_size_chunking, chunk_size=300, overlap=50)
)
print(f"Fixed 300: {len(df_fixed_300)}")

df_fixed_600 = create_chunked_df(
    df_corpus, partial(fixed_size_chunking, chunk_size=600, overlap=100)
)
print(f"Fixed 600: {len(df_fixed_600)}")

df_recursive = create_chunked_df(
    df_corpus, partial(recursive_character_text_splitter, max_length=500)
)
print(f"Recursive: {len(df_recursive)}")

df_semantic = create_chunked_df(
    df_corpus, partial(semantic_sentence_chunker, max_length=500, min_length=100)
)
print(f"Semantic: {len(df_semantic)}")

df_sliding = create_chunked_df(
    df_corpus, partial(sliding_window_chunker, chunk_size=500, overlap_ratio=0.2)
)
print(f"Sliding window: {len(df_sliding)}")

df_hybrid = create_chunked_df(
    df_corpus, partial(hybrid_chunker, target_size=400, max_size=600, min_size=100)
)
print(f"Hybrid: {len(df_hybrid)}")

df_late = create_chunked_df(
    df_corpus, partial(late_chunking, chunk_size=500, context_window=100)
)
print(f"Late chunking: {len(df_late)}")

Creating chunked dataframes...
Sentence chunks: 7568
Fixed 300: 8064
Fixed 600: 4189
Recursive: 4617
Semantic: 4679
Sliding window: 5330
Hybrid: 6153
Late chunking: 5694


In [10]:
# create indexes and compute metrics for these chunking strategies

chunked_dfs = [
    # Original
    ("Sentence chunking", df_sentence_chunks),
    ("Fixed size 300", df_fixed_300),
    ("Fixed size 600", df_fixed_600),
    ("Recursive 500", df_recursive),
    # Improved
    ("Semantic 500", df_semantic),
    ("Sliding window 500", df_sliding),
    ("Hybrid 400-600", df_hybrid),
    ("Late chunking 500", df_late),
]
all_metrics = {}
for df_chunk_name, df_chunk in chunked_dfs:
    index_ = create_nmslib_index(
        get_embeddings(df_chunk['Text_Content'].tolist(), model=emb_model2)
    )
    retriever = create_bm25_retriever(df_chunk['Text_Content'].tolist())

    bm25_search_results = {}
    ann_search_results = {}

    k_neighbors = 10
    
    for query in tqdm(df['Question'].tolist()):
        indices = bm25_search(query, retriever, top_k=k_neighbors)
        bm25_search_results[query] = {'indices': indices}
    
    for query in tqdm(df['Question'].tolist(), desc="ANN Search Model 2"):
        indices, distances = ann_search(query, index_, emb_model2, top_k=k_neighbors)
        ann_search_results[query] = {'indices': indices, 'distances': distances}
    
    print("\n" + "="*60 + "\n")
    print(f"=== Metrics for chunking strategy: {df_chunk_name} ===")
    metrics = compute_metrics(ann_search_results, bm25_search_results, df, df_chunk, number_pages_doc)
    all_metrics[df_chunk_name] = metrics

ANN Search Model 2: 100%|██████████| 461/461 [00:03<00:00, 137.41it/s]




=== Metrics for chunking strategy: Sentence chunking ===
Performance for documents retrieval: 
ANN Recall@1: 0.789587852494577 , BM25 Recall@1: 0.6355748373101953
ANN Recall@3: 0.8806941431670282 , BM25 Recall@3: 0.7852494577006508
ANN Recall@5: 0.911062906724512 , BM25 Recall@5: 0.8286334056399133
ANN Recall@10: 0.9457700650759219 , BM25 Recall@10: 0.8741865509761388

Performance for pages retrieval: 
ANN Page Score: 0.6929366340061435, BM25 Page Score: 0.5547964624672952


ANN Search Model 2: 100%|██████████| 461/461 [00:03<00:00, 144.05it/s]




=== Metrics for chunking strategy: Fixed size 300 ===
Performance for documents retrieval: 
ANN Recall@1: 0.8308026030368764 , BM25 Recall@1: 0.6811279826464208
ANN Recall@3: 0.8937093275488069 , BM25 Recall@3: 0.8134490238611713
ANN Recall@5: 0.9197396963123644 , BM25 Recall@5: 0.8459869848156182
ANN Recall@10: 0.9414316702819957 , BM25 Recall@10: 0.911062906724512

Performance for pages retrieval: 
ANN Page Score: 0.7233225981970938, BM25 Page Score: 0.5942887272777404


ANN Search Model 2: 100%|██████████| 461/461 [00:03<00:00, 147.35it/s]


...



=== Metrics for chunking strategy: Sliding window 500 ===
Performance for documents retrieval: 
ANN Recall@1: 0.8351409978308026 , BM25 Recall@1: 0.7006507592190889
ANN Recall@3: 0.913232104121475 , BM25 Recall@3: 0.7982646420824295
ANN Recall@5: 0.9262472885032538 , BM25 Recall@5: 0.8546637744034707
ANN Recall@10: 0.9457700650759219 , BM25 Recall@10: 0.9023861171366594

Performance for pages retrieval: 
ANN Page Score: 0.7354060798474853, BM25 Page Score: 0.6204099186897106


ANN Search Model 2: 100%|██████████| 461/461 [00:03<00:00, 152.56it/s]




=== Metrics for chunking strategy: Hybrid 400-600 ===
Performance for documents retrieval: 
ANN Recall@1: 0.8156182212581344 , BM25 Recall@1: 0.648590021691974
ANN Recall@3: 0.8937093275488069 , BM25 Recall@3: 0.7722342733188721
ANN Recall@5: 0.9088937093275488 , BM25 Recall@5: 0.8264642082429501
ANN Recall@10: 0.9501084598698482 , BM25 Recall@10: 0.8893709327548807

Performance for pages retrieval: 
ANN Page Score: 0.7183624368006767, BM25 Page Score: 0.566119385056983


ANN Search Model 2: 100%|██████████| 461/461 [00:03<00:00, 144.79it/s]




=== Metrics for chunking strategy: Late chunking 500 ===
Performance for documents retrieval: 
ANN Recall@1: 0.8568329718004338 , BM25 Recall@1: 0.7114967462039046
ANN Recall@3: 0.89587852494577 , BM25 Recall@3: 0.8026030368763557
ANN Recall@5: 0.928416485900217 , BM25 Recall@5: 0.8438177874186551
ANN Recall@10: 0.9566160520607375 , BM25 Recall@10: 0.89587852494577

Performance for pages retrieval: 
ANN Page Score: 0.7516061769535567, BM25 Page Score: 0.6348064319871667


In [11]:
results_df = pd.DataFrame(all_metrics).T
results_df.loc['Baseline'] = metrics_baseline
results_df = results_df.round(4)
results_df = results_df.sort_values('bm25_recall@1', ascending=False)
results_df

,ann_recall@1,ann_recall@3,ann_recall@5,ann_recall@10,bm25_recall@1,bm25_recall@3,bm25_recall@5,bm25_recall@10,ann_page_score,bm25_page_score
Baseline,0.8764,0.9414,0.9653,0.9783,0.7657,0.8568,0.8894,0.9349,0.7419,0.6682
Late chunking 500,0.8568,0.8959,0.9284,0.9566,0.7115,0.8026,0.8438,0.8959,0.7516,0.6348
Fixed size 600,0.8764,0.9219,0.9414,0.9566,0.7050,0.8113,0.8612,0.9154,0.7697,0.6163
Sliding window 500,0.8351,0.9132,0.9262,0.9458,0.7007,0.7983,0.8547,0.9024,0.7354,0.6204
Fixed size 300,0.8308,0.8937,0.9197,0.9414,0.6811,0.8134,0.8460,0.9111,0.7233,0.5943
Recursive 500,0.8395,0.8937,0.9262,0.9501,0.6790,0.8048,0.8655,0.9132,0.7413,0.5941
Semantic 500,0.8265,0.9132,0.9393,0.9610,0.6768,0.7852,0.8482,0.9024,0.7231,0.5982
Hybrid 400-600,0.8156,0.8937,0.9089,0.9501,0.6486,0.7722,0.8265,0.8894,0.7184,0.5661
Sentence chunking,0.7896,0.8807,0.9111,0.9458,0.6356,0.7852,0.8286,0.8742,0.6929,0.5548


In [12]:
#CHECK if something is better than baseline on at least one metric
better_than_baseline = results_df > results_df.loc['Baseline']
better_than_baseline

,ann_recall@1,ann_recall@3,ann_recall@5,ann_recall@10,bm25_recall@1,bm25_recall@3,bm25_recall@5,bm25_recall@10,ann_page_score,bm25_page_score
Baseline,False,False,False,False,False,False,False,False,False,False
Late chunking 500,False,False,False,False,False,False,False,False,True,False
Fixed size 600,False,False,False,False,False,False,False,False,True,False
Sliding window 500,False,False,False,False,False,False,False,False,False,False
Fixed size 300,False,False,False,False,False,False,False,False,False,False
Recursive 500,False,False,False,False,False,False,False,False,False,False
Semantic 500,False,False,False,False,False,False,False,False,False,False
Hybrid 400-600,False,False,False,False,False,False,False,False,False,False
Sentence chunking,False,False,False,False,False,False,False,False,False,False


We can see that chunking does not seem to improve retrieval, which is quite strange but that's what we see from the numbers. Maybe some implementations of chunking are wrong, but I assume that not all of them, so this seems to be valid. Let's test LLM answers with one of the chunking methods.

# 4. Prompt Refinement

In [18]:
df_fixed_600 = create_chunked_df(
    df_corpus, partial(fixed_size_chunking, chunk_size=600, overlap=100)
)

index_ = create_nmslib_index(
    get_embeddings(df_fixed_600['Text_Content'].tolist(), model=emb_model2)
)
retriever = create_bm25_retriever(df_fixed_600['Text_Content'].tolist())

bm25_search_results = {}
ann_search_results = {}

k_neighbors = 10

for query in tqdm(df['Question'].tolist()):
    indices = bm25_search(query, retriever, top_k=k_neighbors)
    bm25_search_results[query] = {'indices': indices}

for query in tqdm(df['Question'].tolist(), desc="ANN Search Model 2"):
    indices, distances = ann_search(query, index_, emb_model2, top_k=k_neighbors)
    ann_search_results[query] = {'indices': indices, 'distances': distances}

ANN Search Model 2: 100%|██████████| 461/461 [00:03<00:00, 144.97it/s]


In [20]:
# check performance of llm using chunked context
RAG_PROMPT = """
You are given a question and a list of context passages. You may use context of passages to answer the question.
As answer respond only with the correct option letter (A, B, etc.).
Context Passages:
{context}
Question: {question}
Options: {options}
Answer (Only option letter):
"""

FEW_SHOT_PROMPT = """
Select appropriate answer for a given question from the list of options.
As answer respond only with the correct option letter (A, B, etc.).

1. Question: {question1}
   Options: {options1}
   Answer: {answer1}

2. Question: {question2}
   Options: {options2}
   Answer: {answer2}

Now, answer the question:
Question: {question}
Options: {options}
Answer:
"""

COT_PROMPT = """
Select the appropriate answer for the given question from the list of options.

Think step by step and explain your reasoning before giving the final answer.

Respond **only** with a JSON object containing the following fields:
{
  "reasoning": "<your step-by-step reasoning>",
  "answer": "<the correct option letter (A, B, etc.)>"
}

Question:
{question}

Options:
{options}

Answer:
"""


In [13]:
def reciprocal_rank_fusion(embedding_ranks, bm25_ranks, alpha=0.5):
    combined_scores = {}
    for rank, idx in enumerate(embedding_ranks):
        combined_scores[idx] = combined_scores.get(idx, 0) + alpha / (rank + 1)
    for rank, idx in enumerate(bm25_ranks):
        combined_scores[idx] = combined_scores.get(idx, 0) + (1 - alpha) / (rank + 1)
    # Sort by combined score
    sorted_indices = sorted(combined_scores.keys(), key=lambda x: combined_scores[x], reverse=True)
    return sorted_indices

In [22]:
indices1 = bm25_search_results[question]['indices']
indices2 = ann_search_results[question]['indices']
indices = reciprocal_rank_fusion(indices2, indices1, alpha=0.5)


In [23]:
answers_rag = []
for _, row in df.iterrows():
    question = row["Question"]
    options = row["options"]
    correct_answer = row["Correct_Answer"]

    # Retrieve context passages
    indices1 = bm25_search_results[question]['indices']
    indices2 = ann_search_results[question]['indices']
    indices = reciprocal_rank_fusion(indices2, indices1, alpha=0.5)[:5]

    context_passages = "\nPASSAGE: \n".join(
        [df_fixed_600.iloc[i]['Text_Content'] for i in indices]
    )

    prompt = RAG_PROMPT.format(question=question, context=context_passages, options=options)
    response = llm.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    
    #choose the first character as answer only
    answer = response.choices[0].message.content.strip()[0]
    answers_rag.append(answer)

#replace rag answer with "A" if it is not in options
for i, row in df.iterrows():
    options_letters = [chr(65 + j) for j in range(6)]  # A-F
    if answers_rag[i] not in options_letters:
        answers_rag[i] = "A"
print(classification_report(answers_rag, df["Correct_Answer"]))

              precision    recall  f1-score   support

           A       0.84      0.75      0.79        88
           B       0.88      0.77      0.82        77
           C       0.79      0.88      0.83        64
           D       0.86      0.86      0.86        77
           E       0.86      0.94      0.90        71
           F       0.87      0.92      0.89        84

    accuracy                           0.85       461
   macro avg       0.85      0.85      0.85       461
weighted avg       0.85      0.85      0.85       461



Around 20% improvement with additional context

In [53]:
question1 = df.iloc[0]["Question"]
options1 = df.iloc[0]["options"]
answer1 = df.iloc[0]["Correct_Answer"]

question2 = df.iloc[1]["Question"]
options2 = df.iloc[1]["options"]
answer2 = df.iloc[1]["Correct_Answer"]

answers_few_shot = []
for _, row in df.iloc[2:].iterrows():
    question = row["Question"]
    options = row["options"]
    correct_answer = row["Correct_Answer"]

    prompt = FEW_SHOT_PROMPT.format(
        question1=question1,
        options1=options1,
        answer1=answer1,
        question2=question2,
        options2=options2,
        answer2=answer2,
        question=question,
        options=options
    )
    response = llm.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    
    #choose the first character as answer only
    answer = response.choices[0].message.content.strip()[0]
    answers_few_shot.append(answer)

#replace answer with "A" if it is not in options
for i in range(len(answers_few_shot)):
    options_letters = [chr(65 + j) for j in range(6)]  # A-F
    if answers_few_shot[i] not in options_letters:
        answers_few_shot[i] = "A"
print(classification_report(answers_few_shot, df.iloc[2:]["Correct_Answer"]))

              precision    recall  f1-score   support

           A       0.58      0.58      0.58        79
           B       0.67      0.56      0.61        78
           C       0.66      0.59      0.63        79
           D       0.61      0.64      0.63        73
           E       0.55      0.65      0.59        65
           F       0.58      0.61      0.60        85

    accuracy                           0.61       459
   macro avg       0.61      0.61      0.61       459
weighted avg       0.61      0.61      0.61       459



Few-show did not improve score at all.

In [ ]:
COT_PROMPT = """
Select the appropriate answer for the given question from the list of options.
Think step by step and explain your reasoning before giving the final answer in 3 sentences max.
Respond ONLY with a JSON object in the following format:
{{
  "reasoning": "<your step-by-step reasoning>",
  "answer": "<the correct option letter (A, B, etc.)>"
}}

Question:
{question}

Options:
{options}

Answer:
"""
import json_repair

answers_cot = []
for _, row in tqdm(df.iterrows()):
    question = row["Question"]
    options = row["options"]
    correct_answer = row["Correct_Answer"]

    prompt = COT_PROMPT.format(question=question, options=options)
    response = llm.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    
    response_json = json_repair.loads(response.choices[0].message.content.strip())
    answer = response_json.get("answer", "A").strip()[0]

    answers_cot.append(answer)
#replace answer with "A" if it is not in options
for i in range(len(answers_cot)):
    options_letters = [chr(65 + j) for j in range(6)]  # A-F
    if answers_cot[i] not in options_letters:
        answers_cot[i] = "A"
print(classification_report(answers_cot, df["Correct_Answer"]))

              precision    recall  f1-score   support

           A       0.62      0.49      0.55        99
           B       0.64      0.54      0.59        80
           C       0.66      0.59      0.63        79
           D       0.62      0.65      0.64        74
           E       0.58      0.75      0.65        60
           F       0.60      0.77      0.67        69

    accuracy                           0.62       461
   macro avg       0.62      0.63      0.62       461
weighted avg       0.62      0.62      0.61       461



# 5. LLM Swap

In [ ]:
from litellm import completion
os.environ["OPENAI_API_KEY"] = API_KEY


answers_rag_openai = []
for _, row in tqdm(df.iterrows()):
    question = row["Question"]
    options = row["options"]
    correct_answer = row["Correct_Answer"]

    # Retrieve context passages
    indices1 = bm25_search_results[question]['indices']
    indices2 = ann_search_results[question]['indices']
    indices = reciprocal_rank_fusion(indices2, indices1, alpha=0.5)[:5]

    context_passages = "\nPASSAGE: \n".join(
        [df_fixed_600.iloc[i]['Text_Content'] for i in indices]
    )

    prompt = RAG_PROMPT.format(question=question, context=context_passages, options=options)
    response = completion("gpt-4o-mini",messages=[{"role": "user", "content": prompt}])

    #choose the first character as answer only
    answer = response.choices[0].message.content.strip()[0]
    answers_rag_openai.append(answer)

#replace rag answer with "A" if it is not in options
for i, row in df.iterrows():
    options_letters = [chr(65 + j) for j in range(6)]  # A-F
    if answers_rag_openai[i] not in options_letters:
        answers_rag_openai[i] = "A"
print(classification_report(answers_rag_openai, df["Correct_Answer"]))


461it [04:39,  1.65it/s]

              precision    recall  f1-score   support

           A       0.87      0.78      0.82        89
           B       0.87      0.76      0.81        76
           C       0.83      0.83      0.83        71
           D       0.83      0.90      0.86        71
           E       0.88      0.96      0.92        72
           F       0.90      0.98      0.94        82

    accuracy                           0.87       461
   macro avg       0.86      0.87      0.86       461
weighted avg       0.87      0.87      0.86       461



Compared to Qwen3-VL-8B-Instruct-FP8 Openai's gpt-4o-mini showed barely any improvements using RAG. 